In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import mlflow
import mlflow.sklearn
import git  # pip install gitpython

# Получаем текущий git коммит автоматически
repo = git.Repo(search_parent_directories=True)
commit_hash = repo.head.commit.hexsha[:7]  # короткий хэш
commit_message = repo.head.commit.message.strip()

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("my_first_experiment")

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

with mlflow.start_run():

    # Параметры
    n_estimators = 100
    max_depth = 5
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # 🔗 Привязка к git — вот где происходит магия!
    mlflow.set_tag("git_commit", commit_hash)
    mlflow.set_tag("git_message", commit_message)

    # Обучение
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
    model.fit(X_train, y_train)

    # Метрики
    accuracy = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric("accuracy", accuracy)

    mlflow.sklearn.log_model(model, "random_forest_model")

    print(f"✅ Accuracy: {accuracy:.2f}")
    print(f"🔗 Git commit: {commit_hash} — '{commit_message}'")

2026/04/26 23:32:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/26 23:32:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Accuracy: 0.97
🔗 Git commit: f622991 — 'baseline: max_depth=3 n_estimators=100'
🏃 View run nebulous-worm-143 at: http://localhost:5000/#/experiments/1/runs/c648218f948a4e6ca20ccfb6340733ca
🧪 View experiment at: http://localhost:5000/#/experiments/1
